## Bonus Exercise - Performance Analysis and Optimization of the Game of Life Code
### B.1 Check the code with a linter, and in case, run an auto-formatter. Produce HTML documentation running sphinx.
First we comment out the visualization and animation part of the game of life code and add replace it with just the execution of the "update"-function, for a fixed number of iterations.
Let's use pylint as linter on the code afterwards:

In [17]:
!pylint conway.py

************* Module conway
conway.py:11:31: C0303: Trailing whitespace (trailing-whitespace)
conway.py:24:38: C0303: Trailing whitespace (trailing-whitespace)
conway.py:25:38: C0303: Trailing whitespace (trailing-whitespace)
conway.py:59:28: C0303: Trailing whitespace (trailing-whitespace)
conway.py:64:70: C0303: Trailing whitespace (trailing-whitespace)
conway.py:66:62: C0303: Trailing whitespace (trailing-whitespace)
conway.py:67:62: C0303: Trailing whitespace (trailing-whitespace)
conway.py:68:74: C0303: Trailing whitespace (trailing-whitespace)
conway.py:95:0: C0303: Trailing whitespace (trailing-whitespace)
conway.py:100:0: C0303: Trailing whitespace (trailing-whitespace)
conway.py:127:18: C0303: Trailing whitespace (trailing-whitespace)
conway.py:80:0: R1707: Disallow trailing comma tuple (trailing-comma-tuple)
conway.py:9:0: C0410: Multiple imports on one line (sys, argparse) (multiple-imports)
conway.py:12:0: R0402: Use 'from matplotlib import animation' instead (consider-usin

Lets use black as auto formatter on the code:

In [21]:
!black conway.py

reformatted conway.py

All done! ✨ 🍰 ✨
1 file reformatted.


The sphinx html doc looks like this:
![conway doc](conway_doc.png)

## Task B.2 Measure the execution time, varying the grid size (and fixed number of iterations). Make a plot with this information.

The results of the different grid sizes and their resulting execution time can be seen here:

![conway exec times](conway_exec_time.png)

The executions time grows exponentially.

## Task B.3 Use different profilers (from coarse- to fine-grained) to identify performance bottlenecks and potential improvement (disregard perf if you don't have access to it). Report the results of the profilers. The choice of profilers is up to you.
In order to profile the code it is again modified to not iterate through different grid sizes anymore. We now use a grid size of N=100 and 1000 iterations.
Lets start with the cProfiler first.

In [27]:
!python -m cProfile -s cumulative conway.py

         1220236 function calls (1202812 primitive calls) in 12.182 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    334/1    0.004    0.000   12.183   12.183 {built-in method builtins.exec}
        1    0.000    0.000   12.183   12.183 conway.py:1(<module>)
        1    0.003    0.003   11.577   11.577 conway.py:96(main)
     1000   11.539    0.012   11.552    0.012 conway.py:60(update)
       44    0.001    0.000    0.831    0.019 __init__.py:1(<module>)
    395/5    0.001    0.000    0.622    0.124 <frozen importlib._bootstrap>:1360(_find_and_load)
    392/5    0.001    0.000    0.622    0.124 <frozen importlib._bootstrap>:1308(_find_and_load_unlocked)
    376/6    0.001    0.000    0.620    0.103 <frozen importlib._bootstrap>:914(_load_unlocked)
    324/6    0.001    0.000    0.620    0.103 <frozen importlib._bootstrap_external>:753(exec_module)
  1099/11    0.001    0.000    0.618    0.056 <frozen importlib._boots

The update functions takes a lot of time, which makes sense since all calculations are happening here. Lets use the line profiler to get the execution times for each row. Therefore we must add a @profile decorator to the code. Since the update function is most interesting, let's decorate this one.

In [29]:
!python -m kernprof -l conway.py
!python -m line_profiler conway.py.lprof

Wrote profile results to 'conway.py.lprof'
Inspect results with:
python -m line_profiler -rmt conway.py.lprof
Timer unit: 1e-06 s

Total time: 53.0133 s
File: conway.py
Function: update at line 59

Line #      Hits         Time  Per Hit   % Time  Line Contents
    59                                           @profile
    60                                           def update(frameNum, img, grid, N):
    61                                               # copy grid since we require 8 neighbors for calculation
    62                                               # and we go line by line
    63      1000      10466.0     10.5      0.0      newGrid = grid.copy()
    64    101000      17902.0      0.2      0.0      for i in range(N):
    65  10100000    1982642.0      0.2      3.7          for j in range(N):
    66                                                       # compute 8-neghbor sum
    67                                                       # using toroidal boundary conditions - 

The line_profiler shows that the most time is used in line 71. The other lines calculation the number of neighbors of each cell (line 71-78) are also taking quite some time. Thats because the numpy arrays are accessed in a python loop which takes a lot more time and therefore should be vectorized. Lets create a new file "conway_optimized.py" in order to optimize the code and compare it to the old one afterwards.

### Task B.4 Implement an optimization, report the new profiling results, and show the performance improvement.

As presented in [python to numpy article](https://www.labri.fr/perso/nrougier/from-python-to-numpy/) it is possible to vectorize this part to get better performance.

Looking at the different execution times for different grid sizes in comparison with the old unoptimized code we can see that we were able to reduce the execution time by quite a lot. So vectorizing the update function made a huge difference in terms of performance.

![conway exec time optimized](conway_optimized_exec_time.png)

Let's also profile the update function again:

In [30]:
!python -m kernprof -l conway_optimized.py
!python -m line_profiler conway_optimized.py.lprof

Wrote profile results to 'conway_optimized.py.lprof'
Inspect results with:
python -m line_profiler -rmt conway_optimized.py.lprof
Timer unit: 1e-06 s

Total time: 0.108663 s
File: conway_optimized.py
Function: update at line 59

Line #      Hits         Time  Per Hit   % Time  Line Contents
    59                                           @profile
    60                                           def update(frameNum, img, grid, N):
    61      1000       1959.0      2.0      1.8      neighbors = np.zeros(grid.shape, dtype=int)
    62      2000      23161.0     11.6     21.3      neighbors[1:-1, 1:-1] += (
    63      6000      45473.0      7.6     41.8          grid[:-2, :-2] + grid[:-2, 1:-1] + grid[:-2, 2:] +
    64      2000        800.0      0.4      0.7          grid[1:-1, :-2]                 + grid[1:-1, 2:] +
    65      3000        973.0      0.3      0.9          grid[2:, :-2] + grid[2:, 1:-1] + grid[2:, 2:]
    66      1000        215.0      0.2      0.2      ) // 255
    67 

Counting the neighbors still takes the most time but was reduced from 18682920.0 to 45473.0 which is >400 times faster.

The update function has been vectorized and now looks like this:
```python
def update(frameNum, img, grid, N):
    neighbors = np.zeros(grid.shape, dtype=int)
    neighbors[1:-1, 1:-1] += (
        grid[:-2, :-2] + grid[:-2, 1:-1] + grid[:-2, 2:] +
        grid[1:-1, :-2]                 + grid[1:-1, 2:] +
        grid[2:, :-2] + grid[2:, 1:-1] + grid[2:, 2:]
    ) // 255
    birth = (grid[1:-1, 1:-1] == OFF) & (neighbors[1:-1, 1:-1] == 3)
    survive = (grid[1:-1, 1:-1] == ON) & ((neighbors[1:-1, 1:-1] == 2) | (neighbors[1:-1, 1:-1] == 3))
    grid[1:-1, 1:-1] = np.where(birth | survive, ON, OFF)
```